# 🚁 Disaster Debris Human Detection — YOLOv8 Training
**Dataset:** disaster.v1i.yolov8 
**Goal:** Detect survivors in disaster/debris scenes from drone

---
### Run cells ONE BY ONE from top to bottom

## Cell 1 — Install Requirements

In [3]:
# Install required packages
import sys
!{sys.executable} -m pip install ultralytics pyyaml --quiet
print('✅ Packages installed!')

✅ Packages installed!


## Cell 2 — Check GPU

In [5]:
import torch

print('=' * 50)
print('  SYSTEM CHECK')
print('=' * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU      : ✅ {gpu_name}')
    print(f'  GPU RAM  : {gpu_mem:.1f} GB')
    DEVICE = 0
    
    # Set batch size based on GPU RAM
    if gpu_mem >= 8:
        BATCH = 16
    elif gpu_mem >= 4:
        BATCH = 8
    else:
        BATCH = 4
else:
    print('  GPU      : ❌ Not found — using CPU (slow!)')
    DEVICE = 'cpu'
    BATCH  = 4

print(f'  Batch size set to: {BATCH}')
print('=' * 50)

  SYSTEM CHECK
  GPU      : ✅ NVIDIA RTX A4000
  GPU RAM  : 17.2 GB
  Batch size set to: 16


## Cell 3 — Set Dataset Path

In [7]:
from pathlib import Path
import yaml
import os

# ══════════════════════════════════════════════
# ★ CHANGE THIS TO YOUR DATASET FOLDER PATH ★
# ══════════════════════════════════════════════
DATASET_ROOT = r"C:\Users\WNAAdmin\Downloads\disaster.v1i.yolov8"

# Check dataset exists
dataset_path = Path(DATASET_ROOT)
yaml_path    = dataset_path / 'data.yaml'

print('Checking dataset structure...')
checks = [
    dataset_path / 'train' / 'images',
    dataset_path / 'train' / 'labels',
    dataset_path / 'valid' / 'images',
    dataset_path / 'valid' / 'labels',
    dataset_path / 'test'  / 'images',
    yaml_path,
]

all_ok = True
for p in checks:
    if p.exists():
        if p.is_dir():
            count = len(list(p.glob('*')))
            print(f'  ✅ {p.name:<10} ({count} files)')
        else:
            print(f'  ✅ {p.name}')
    else:
        print(f'  ❌ MISSING: {p}')
        all_ok = False

if all_ok:
    print('\n✅ Dataset structure OK!')
else:
    print('\n❌ Fix missing paths before continuing!')

Checking dataset structure...
  ✅ images     (4284 files)
  ✅ labels     (4284 files)
  ✅ images     (1226 files)
  ✅ labels     (1226 files)
  ✅ images     (614 files)
  ✅ data.yaml

✅ Dataset structure OK!


## Cell 4 — Fix data.yaml Paths

In [9]:
# Fix relative paths in data.yaml to absolute paths
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

print('Original data.yaml:')
for k, v in data.items():
    print(f'  {k}: {v}')

# Update to absolute path
data['path'] = str(dataset_path.resolve())

# Save fixed yaml
fixed_yaml = dataset_path / 'data_fixed.yaml'
with open(fixed_yaml, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f'\nFixed data.yaml:')
for k, v in data.items():
    print(f'  {k}: {v}')

print(f'\n✅ Fixed yaml saved → {fixed_yaml}')

Original data.yaml:
  path: .
  train: train/images
  val: valid/images
  test: test/images
  nc: 1
  names: ['person']

Fixed data.yaml:
  path: C:\Users\WNAAdmin\Downloads\disaster.v1i.yolov8
  train: train/images
  val: valid/images
  test: test/images
  nc: 1
  names: ['person']

✅ Fixed yaml saved → C:\Users\WNAAdmin\Downloads\disaster.v1i.yolov8\data_fixed.yaml


## Cell 5 — Training Configuration

In [25]:
# ══════════════════════════════════════════════
# TRAINING CONFIG — change if needed
# ══════════════════════════════════════════════

BASE_MODEL   = 'yolov8n.pt'          # base model
EPOCHS       = 100                    # training epochs
IMGSZ        = 960                    # image size
WORKERS      = 0                      # 0 = Windows safe
PROJECT_DIR  = 'runs/rescue'
RUN_NAME     = 'disaster_detection'
CONF_THRESH  = 0.35                   # low = catches more in debris
IOU_THRESH   = 0.45

print('=' * 50)
print('  TRAINING CONFIGURATION')
print('=' * 50)
print(f'  Base model  : {BASE_MODEL}')
print(f'  Epochs      : {EPOCHS}')
print(f'  Image size  : {IMGSZ}')
print(f'  Batch size  : {BATCH}')
print(f'  Device      : {DEVICE}')
print(f'  Conf thresh : {CONF_THRESH}')
print(f'  Output      : {PROJECT_DIR}/{RUN_NAME}')
print('=' * 50)
print('\n✅ Config ready — run next cell to start training!')

  TRAINING CONFIGURATION
  Base model  : yolov8n.pt
  Epochs      : 100
  Image size  : 960
  Batch size  : 16
  Device      : 0
  Conf thresh : 0.35
  Output      : runs/rescue/disaster_detection

✅ Config ready — run next cell to start training!


## Cell 6 — ⚡ START TRAINING
> This will take **1-3 hours** depending on your GPU. Watch mAP50 score climb each epoch — aim for > 0.70

In [27]:
from ultralytics import YOLO
from datetime import datetime

# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('✅ GPU cache cleared')

# Load base model
print(f'Loading base model: {BASE_MODEL}...')
model = YOLO(BASE_MODEL)
print('✅ Model loaded\n')

print('🚀 TRAINING STARTED...')
print('Watch mAP50 — aim for > 0.70\n')

start_time = datetime.now()

results = model.train(
    data         = str(fixed_yaml),

    # Core
    epochs       = EPOCHS,
    imgsz        = IMGSZ,
    batch        = BATCH,
    workers      = WORKERS,
    device       = DEVICE,

    # Output
    project      = PROJECT_DIR,
    name         = RUN_NAME,
    exist_ok     = True,

    # Optimizer
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,

    # Detection
    conf         = CONF_THRESH,
    iou          = IOU_THRESH,

    # Augmentation — important for disaster scenes!
    augment      = True,
    degrees      = 10.0,   # rotation — drone views at angles
    translate    = 0.1,
    scale        = 0.5,    # scale — different drone heights
    fliplr       = 0.5,    # horizontal flip
    mosaic       = 1.0,    # mosaic augmentation
    hsv_h        = 0.015,  # color variation
    hsv_s        = 0.7,
    hsv_v        = 0.4,

    # Performance
    amp          = True,   # mixed precision — faster
    cache        = True,   # cache images in RAM

    # Logging
    verbose      = True,
    plots        = True,
    save         = True,
    save_period  = 10,     # save every 10 epochs
)

elapsed = datetime.now() - start_time

print('\n' + '=' * 50)
print('  ✅ TRAINING COMPLETE!')
print('=' * 50)
print(f'  Time taken : {elapsed}')
print(f'  Best model : {PROJECT_DIR}/{RUN_NAME}/weights/best.pt')
print('=' * 50)

✅ GPU cache cleared
Loading base model: yolov8n.pt...
✅ Model loaded

🚀 TRAINING STARTED...
Watch mAP50 — aim for > 0.70

New https://pypi.org/project/ultralytics/8.4.39 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.35  Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA RTX A4000, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.35, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\WNAAdmin\Downloads\disaster.v1i.yolov8\data_fixed.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.45, keras=False, kobj=1.0, line_width=


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\WNAAdmin\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\WNAAdmin\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\WNAAdmin\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\WNAAdmin\anaconda3\L

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## Cell 7 — Validate Model Accuracy

In [ ]:
best_model_path = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'

print('Running validation on test set...')
model_val = YOLO(best_model_path)

metrics = model_val.val(
    data    = str(fixed_yaml),
    imgsz   = IMGSZ,
    conf    = CONF_THRESH,
    iou     = IOU_THRESH,
    device  = DEVICE,
    verbose = True,
)

print('\n' + '=' * 50)
print('  VALIDATION RESULTS')
print('=' * 50)
print(f'  mAP50     : {metrics.box.map50:.3f}  (aim > 0.70)')
print(f'  mAP50-95  : {metrics.box.map:.3f}')
print(f'  Precision : {metrics.box.mp:.3f}')
print(f'  Recall    : {metrics.box.mr:.3f}')

if metrics.box.map50 >= 0.70:
    print('\n  ✅ Excellent! Ready for drone deployment!')
elif metrics.box.map50 >= 0.50:
    print('\n  ⚠️  Decent. Consider more epochs.')
else:
    print('\n  ❌ Low accuracy. Increase epochs to 150.')
print('=' * 50)

## Cell 8 — Test On Sample Disaster Images

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Get 5 test images
test_img_dir = dataset_path / 'test' / 'images'
test_images  = list(test_img_dir.glob('*.jpg'))[:5]
test_images += list(test_img_dir.glob('*.png'))[:5]
test_images  = test_images[:5]

print(f'Testing on {len(test_images)} sample images...\n')

model_test = YOLO(best_model_path)

fig, axes = plt.subplots(1, len(test_images), figsize=(20, 5))
if len(test_images) == 1:
    axes = [axes]

for i, img_path in enumerate(test_images):
    result = model_test.predict(
        source  = str(img_path),
        imgsz   = IMGSZ,
        conf    = CONF_THRESH,
        iou     = IOU_THRESH,
        device  = DEVICE,
        verbose = False,
    )[0]

    # Draw detections
    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    n_det = len(result.boxes)
    axes[i].imshow(annotated)
    axes[i].set_title(f'Persons: {n_det}', fontsize=12,
                      color='green' if n_det > 0 else 'red')
    axes[i].axis('off')

    print(f'  Image {i+1}: {n_det} person(s) detected')

plt.suptitle('Disaster Scene Detection Results', fontsize=14)
plt.tight_layout()
plt.savefig('test_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Results saved → test_results.png')